# `sc_llm_x402` Buyer Notebook — real end-to-end chat payment (Deno/TypeScript)

Drives the **real, locally-running** `sc_llm_x402.ts` batch-settlement chat handler as a buyer would —
the missing real-server verification for Phase B, and a literal, runnable blueprint for
`website/hooks/useX402Chat.ts` (Phase C, not yet built). Unlike
`x402_facilitator/notebooks/x402_batch_settlement_buyer.ipynb` (which hits the facilitator's raw
`/verify`+`/settle` endpoints directly with hand-built payment requirements), this notebook only ever
talks to the **resource server** over plain HTTP via `wrapFetchWithPayment` — exactly what a real client
(browser or otherwise) does. The server decides and advertises everything else.

**Terminology — x402 SDK role → this repo's package:**

| SDK role | Repo package | Role in this repo |
|---|---|---|
| **Client** (buyer / payer) | `website/` *(this notebook stands in for it)* | Browser wallet: signs the deposit + per-message vouchers |
| **Server** (seller / merchant) | `scw_js/` (`sc_llm_x402.ts`) | Verifies vouchers, serves the LLM response, commits the charge |
| **Facilitator** | `x402_facilitator/` | Neither buyer nor seller — executes on-chain deposit/claim/settle |

## ⚠️ Prerequisites

1. **Deno Jupyter kernel** — `deno jupyter --install` (same global kernel as the sibling facilitator notebooks).
2. **`scw_js/.env`** needs:
   - `TEST_WALLET_PRIVATE_KEY` — the buyer wallet, funded with Base Sepolia USDC (get some from
     https://faucet.circle.com/). Purely a notebook-testing convenience — no scw_js production code
     reads this key.
   - `NFT_WALLET_PUBLIC_KEY` — the receiver address (the server reads the same file).
   - `RECEIVER_AUTHORIZER_PRIVATE_KEY` — required for `createLLMResourceServer()` to construct at all.
     A pure off-chain signer, no funding needed. See `assistent_plan.md` §Backlog F for why this isn't
     delegated to the facilitator (yet).
   - `SCW_ACCESS_KEY` / `SCW_SECRET_KEY` — the server's `S3ChannelStorage` writes real objects under
     `channels/` in the production bucket when this runs (private ACL, harmless, but real).
3. **Start the local server** (separate terminal): `cd scw_js && npm run dev:llmx402` — listens on `:8085`,
   and internally talks to the **real deployed facilitator** (`https://facilitator.fretchen.eu` by default)
   unless `FACILITATOR_URL` is overridden. This means the first cell that opens a channel submits a
   **real on-chain transaction on Base Sepolia**.

In [1]:
// Setup: imports + env
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";

// scw_js's own .env, one level up — the single source of truth (buyer key, receiver
// address, and the server's own secrets all live here; the server reads the same file).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error("TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with Base Sepolia USDC.");
}
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("🚀 sc_llm_x402 buyer notebook");
console.log(`   Buyer (payer): ${account.address}`);
console.log(`   Receiver     : ${env.NFT_WALLET_PUBLIC_KEY ?? "(not set — server will 500)"}`);

🚀 sc_llm_x402 buyer notebook
   Buyer (payer): 0x553179556FC2A39e535D65b921e01fA995E79101
   Receiver     : 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Network selection

Base Sepolia only — the canonical `BATCH_SETTLEMENT_ADDRESS` has no deployment on Optimism Sepolia
(confirmed in the facilitator's own buyer spike). `getBatchSettlementNetworks()` on the server side
already restricts to Base Sepolia / Base mainnet / Optimism mainnet; this notebook only ever offers
the testnet.


In [2]:
const NETWORK = "eip155:84532"; // Base Sepolia
const SERVICE_URL = "http://localhost:8085"; // sc_llm_x402.ts local dev server (npm run dev:llmx402)

console.log(`🧪 Base Sepolia (${NETWORK}) — service: ${SERVICE_URL}`);

🧪 Base Sepolia (eip155:84532) — service: http://localhost:8085


## Buyer setup — the shape `useX402Chat.ts` will need

Unlike the `exact` scheme (`registerExactEvmScheme` helper), batch-settlement has **no client-register
helper** — the scheme is constructed and registered manually, same as the facilitator's buyer spike.

For channel storage this notebook uses a tiny `WebStorageClientChannelStorage` adapter over Deno's
**global `localStorage`** — the same Web Storage API a browser exposes, and (unlike
`InMemoryClientChannelStorage`) it persists across kernel restarts, so an open channel is reused on a
re-run exactly as it must survive a page reload in production. `useX402Chat.ts` reuses this *same* class,
passing `window.localStorage` instead of Deno's global (per `assistent_plan.md` Phase C's plan).
Everything else below is copy-paste-equivalent to what the hook needs.


In [3]:
import { x402Client, wrapFetchWithPayment, x402HTTPClient } from "npm:@x402/fetch@^2.17.0";
import { toClientEvmSigner } from "npm:@x402/evm@^2.17.0";
import {
    BatchSettlementEvmScheme,
    type ClientChannelStorage,
    type BatchSettlementClientContext,
} from "npm:@x402/evm@^2.17.0/batch-settlement/client";
import { createPublicClient, http } from "npm:viem@2";
import { baseSepolia } from "npm:viem@2/chains";

// readContract is documented as "optional" on ClientEvmSigner (only "required for extension
// enrichment" per the SDK's own JSDoc), but batch-settlement's corrective-402 recovery
// (processCorrectivePaymentRequired, triggered on invalid_batch_settlement_evm_cumulative_amount_mismatch
// and _below_claimed) unconditionally needs it too — both recoverFromSignature and
// recoverFromOnChainState bail out immediately with `if (!deps.signer.readContract) return false`.
// Without it, a client/server cumulative-total desync (e.g. from an old channel reused across
// notebook/kernel restarts via localStorage) surfaces as a hard 402 instead of self-healing.
// toClientEvmSigner() is the SDK's own helper for composing a full signer from a plain
// account + a public client. In useX402Chat.ts: swap `account`/`publicClient` here for a wagmi
// WalletClient adapter + `usePublicClient()` (see useX402ImageGeneration.ts's signer shape).
const publicClient = createPublicClient({ chain: baseSepolia, transport: http() });
const buyerSigner = toClientEvmSigner(
    { address: account.address, signTypedData: (a: any) => account.signTypedData(a) },
    publicClient,
);

// ClientChannelStorage backed by the Web Storage API. Channel context is all strings, so
// plain JSON round-trips. In useX402Chat.ts, construct this SAME class with window.localStorage.
class WebStorageClientChannelStorage implements ClientChannelStorage {
    constructor(private backend: Storage, private prefix = "x402-channel:") {}
    private keyFor(key: string) { return `${this.prefix}${key.toLowerCase()}`; }
    async get(key: string) {
        const raw = this.backend.getItem(this.keyFor(key));
        return raw ? (JSON.parse(raw) as BatchSettlementClientContext) : undefined;
    }
    async set(key: string, context: BatchSettlementClientContext) {
        this.backend.setItem(this.keyFor(key), JSON.stringify(context));
    }
    async delete(key: string) { this.backend.removeItem(this.keyFor(key)); }
}

// Deno exposes the global `localStorage` (persists across kernel restarts). Browser: window.localStorage.
const buyerStorage = new WebStorageClientChannelStorage(localStorage);
const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage });

const client = new x402Client();
client.register(NETWORK, buyerScheme);

const fetchWithPayment = wrapFetchWithPayment(fetch, client);
console.log("✅ buyer client ready, registered for", NETWORK);

✅ buyer client ready, registered for eip155:84532


## Message #1 — opens the channel (real on-chain deposit tx)

The first call to `fetchWithPayment` against a resource it hasn't paid for yet: an initial bare request
gets `402`, the SDK builds a **deposit** payload (open the channel + first voucher) and signs it, the
retry carries the payment header, and the server verifies + settles (the deposit — the one real on-chain
tx per channel lifetime, confirmed in the B0 spike). Extracting the settlement receipt mirrors
`useX402ImageGeneration.ts`'s `x402HTTPClient.getPaymentSettleResponse(...)` call exactly.


In [4]:
async function sendChatMessage(content: string) {
    const started = performance.now();

    // FIXED (x402_facilitator PR #545, confirmed deployed — /supported now reports the real
    // signer address, shipped in the same deploy): x402_facilitator's settlePayment() used to
    // rebuild its own SettleResult and drop the SDK's `extra` (incl. channelState.channelId),
    // so a real, correctly-settled payment's response reached the client missing channelId and
    // crashed processSettleResponse() on channelId.toLowerCase(). That crash — while it was still
    // live — could leave a channel settled server-side but never persisted client-side, which is
    // the likely origin of any lingering cumulative-amount desync on a reused channel (see the
    // `toClientEvmSigner` fix in the previous cell for why that no longer surfaces as a hard
    // failure either way). Keeping this try/catch as defense-in-depth, not because the crash is
    // expected anymore.
    let response: Response;
    try {
        response = await fetchWithPayment(SERVICE_URL, {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ data: { prompt: [{ role: "user", content }], useDummyData: true } }),
        });
    } catch (err) {
        const elapsedMs = Math.round(performance.now() - started);
        console.log(`📡 fetchWithPayment THREW after ${elapsedMs}ms:`, (err as Error).message);
        console.log("   (server may have settled successfully anyway — check the server's own log)");
        return { response: null, body: null, receipt: null, elapsedMs, threw: (err as Error).message };
    }
    const elapsedMs = Math.round(performance.now() - started);

    const bodyText = await response.text();
    let body: any;
    try { body = JSON.parse(bodyText); } catch { body = bodyText; }

    console.log(`📡 status=${response.status}  elapsed=${elapsedMs}ms`);
    console.log("📨 body:", JSON.stringify(body, null, 2).slice(0, 600));

    let receipt: unknown = null;
    try {
        const httpClient = new x402HTTPClient(client);
        receipt = httpClient.getPaymentSettleResponse((name: string) => response.headers.get(name));
        console.log("🧾 settlement receipt:", JSON.stringify(receipt, null, 2));
    } catch (err) {
        console.log("🧾 no settlement receipt:", (err as Error).message);
    }
    return { response, body, receipt, elapsedMs, threw: null };
}

const msg1 = await sendChatMessage("What is the capital of France?");

📡 status=200  elapsed=11569ms
📨 body: {
  "content": "I am a placeholder for the LLM response",
  "usage": {
    "prompt_tokens": 5,
    "completion_tokens": 15,
    "total_tokens": 15
  },
  "model": "placeholder model"
}
🧾 settlement receipt: {
  "success": true,
  "payer": "0x553179556fc2a39e535d65b921e01fa995e79101",
  "transaction": "",
  "network": "eip155:84532",
  "amount": "",
  "extra": {
    "channelState": {
      "channelId": "0xdd9e576d5d30096bce8ed29916ee2d3faaf3a34269011b881eccfb0e082719d7",
      "balance": "0",
      "totalClaimed": "0",
      "withdrawRequestedAt": 0,
      "refundNonce": "0",
      "chargedCumulativeAmount": "2840"
    },
    "chargedAmount": "1420"
  }
}


## Messages #2 and #3 — same channel, the actual spike question

**What we don't know yet, and this cell exists to find out:** does `fetchWithPayment` (via
`client.createPaymentPayload()`, called internally on every `402`) automatically notice
`buyerStorage` already holds an open channel for this `(receiver, network, asset)` and sign a plain
**voucher** against it (zero on-chain tx — the actual batching payoff) — or does it try to open a
**second, wasteful deposit**? The facilitator's buyer spike only demonstrated channel reuse via the
low-level `signVoucher` helper called *manually*, never through the automatic `fetchWithPayment` path —
so this genuinely hasn't been observed yet. Whatever happens here is what `useX402Chat.ts` needs to
account for.


In [5]:
const msg2 = await sendChatMessage("And what is the capital of Germany?");
const msg3 = await sendChatMessage("And Italy?");

📡 status=200  elapsed=9565ms
📨 body: {
  "content": "I am a placeholder for the LLM response",
  "usage": {
    "prompt_tokens": 5,
    "completion_tokens": 15,
    "total_tokens": 15
  },
  "model": "placeholder model"
}
🧾 settlement receipt: {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x0b2e109339ae6f7995c086d53248697df90a24099683e20cf5d02866cfdec791",
  "network": "eip155:84532",
  "extra": {
    "channelState": {
      "channelId": "0xdd9e576d5d30096bce8ed29916ee2d3faaf3a34269011b881eccfb0e082719d7",
      "balance": "21300",
      "totalClaimed": "0",
      "withdrawRequestedAt": 0,
      "refundNonce": "0",
      "chargedCumulativeAmount": "4260"
    },
    "chargedAmount": "1420"
  }
}
📡 status=200  elapsed=1403ms
📨 body: {
  "content": "I am a placeholder for the LLM response",
  "usage": {
    "prompt_tokens": 5,
    "completion_tokens": 15,
    "total_tokens": 15
  },
  "model": "placeholder model"
}
🧾 settlement receipt: {


## Findings (fill in after running the cells above for real)

_Placeholder — replace with what was actually observed running against the live local server +
real Base Sepolia facilitator. Do not fill this in from assumption; see `assistent_plan.md`'s
'verify before assuming' discipline used throughout this project._

- [ ] Message #1: real deposit tx confirmed? Explorer link:
- [ ] Message #1: `chargedCumulativeAmount` after settlement (compare to `LLM_ESTIMATED_TOKENS_PER_MESSAGE`
  → `convertTokensToUsdcCost` expectation):
- [ ] Messages #2/#3: did `fetchWithPayment` reuse the channel (voucher, fast, no new 402/deposit) or
  open a new one each time?
- [ ] If reuse did **not** happen automatically: what does `useX402Chat.ts` need to do instead
  (e.g. check `buyerStorage`/`localStorage` for an existing channel before calling `fetchWithPayment`,
  or call `signVoucher` directly)?
